In [4]:
import pandas as pd
import numpy as np
from datetime import datetime
import re
import ast
import json
import os
from tqdm.notebook import tqdm  # or from tqdm import tqdm


# Translation Table

In [ ]:
# datasets from Translation_02_Drug_Disease

In [13]:
# for the old analysis translation_table_drug_disease_OLD.csv
translation_table_full = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_FULL.csv") # from Translation_02_Drug_Disease
translation_table_full.shape

(78107, 29)

In [14]:
fda_only_translated = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_neuro_FDA_only.csv")
fda_only_translated['fda_only']=True
fda_only_translated.shape

(531, 14)

In [15]:
translation_table = pd.concat([translation_table_full, fda_only_translated], ignore_index=True)

In [16]:
n_diseases = translation_table["merged_mondo_label"].nunique()
n_drugs = translation_table["merged_umls_label"].nunique()

print("\n=== Entity coverage summary ===")
print(f"Translation table shape: {translation_table.shape,}")
print(f"Total unique diseases: {n_diseases:,}")
print(f"Total unique drugs: {n_drugs:,}")


=== Entity coverage summary ===
Translation table shape: ((78638, 30),)
Total unique diseases: 5,079
Total unique drugs: 10,100


In [25]:
translation_table.head()

,normalized_key,clinical_doc_ids,merged_mondo_termid,merged_umls_termid,merged_mondo_label,merged_umls_label,trial_start_year,trial_completion_year,phase,overall_status,...,trial_completion_year_first_completed_phase3,fda_merged_umls_label,fda_disease_mondo_term_norm,fda_earliest_year,fda_disease,fda_documents,fda_disease<>drug,fda_AP,fda_only,matched_term
10,obsolete disorder involving pain <> bupivacaine,"['NCT00001724', 'NCT00008476', 'NCT00050362', ...",MONDO:0021668,C0006400,obsolete disorder involving pain,bupivacaine,"[1997.0, 2001.0, 2002.0, 1998.0, 2005.0, 2006....","[2001.0, 2004.0, 2004.0, 2000.0, 2013.0, 2006....","['PHASE2', 'PHASE2', 'PHASE2', 'PHASE3', 'PHAS...","['COMPLETED', 'COMPLETED', 'COMPLETED', 'COMPL...",...,2000.0,Bupivacaine,obsolete disorder involving pain,1981.0,pain,"['ANDA070552', 'ANDA070553', 'ANDA070966', 'AN...",obsolete disorder involving pain <> bupivacaine,True,NaN,obsolete disorder involving pain
12,obsolete disorder involving pain <> lidocaine,"['NCT00001724', 'NCT00006070', 'NCT00006299', ...",MONDO:0021668,C0023660,obsolete disorder involving pain,lidocaine,"[1997.0, 2000.0, 1999.0, 2001.0, 2001.0, 2002....","[2001.0, 2004.0, 2004.0, 2004.0, 2003.0, 2004....","['PHASE2', 'PHASE2', 'PHASE2', 'PHASE2', 'PHAS...","['COMPLETED', 'COMPLETED', 'COMPLETED', 'COMPL...",...,2004.0,Lidocaine,obsolete disorder involving pain,1999.0,pain,"['ANDA076453', 'ANDA202346', 'ANDA206297', 'AN...",obsolete disorder involving pain <> lidocaine,True,NaN,obsolete disorder involving pain
15,obsolete disorder involving pain <> morphine,"['NCT00003687', 'NCT00004390', 'NCT00004696', ...",MONDO:0021668,C0026549,obsolete disorder involving pain,morphine,"[1998.0, 1995.0, 1994.0, 2000.0, 2001.0, 2001....","[2009.0, nan, 1998.0, 2003.0, 2006.0, 2004.0, ...","['PHASE3', 'PHASE3', nan, 'PHASE1/PHASE2', 'PH...","['COMPLETED', 'COMPLETED', 'COMPLETED', 'COMPL...",...,2001.0,Morphine,obsolete disorder involving pain,1984.0,pain,"['ANDA073509', 'ANDA073510', 'ANDA078761', 'AN...",obsolete disorder involving pain <> morphine,True,NaN,obsolete disorder involving pain
25,obsolete disorder involving pain <> acetaminophen,"['NCT00000425', 'NCT00000731', 'NCT00006070', ...",MONDO:0021668,C0000970,obsolete disorder involving pain,acetaminophen,"[1996.0, nan, 2000.0, 1999.0, 2001.0, 2002.0, ...","[2001.0, 1990.0, 2004.0, 2004.0, 2004.0, 2004....","['PHASE3', nan, 'PHASE2', 'PHASE2', 'PHASE2', ...","['COMPLETED', 'COMPLETED', 'COMPLETED', 'COMPL...",...,2001.0,Acetaminophen,obsolete disorder involving pain,1980.0,pain; pains,"['ANDA040330', 'ANDA040400', 'ANDA040405', 'AN...",obsolete disorder involving pain <> acetaminophen,True,NaN,obsolete disorder involving pain
39,glioblastoma <> temozolomide,"['NCT00003464', 'NCT00004200', 'NCT00006353', ...",MONDO:0018177,C0076080,glioblastoma,temozolomide,"[1997.0, 1999.0, 2000.0, 2000.0, 2002.0, 2001....","[2003.0, nan, nan, 2006.0, 2012.0, 2005.0, nan...","['PHASE2', 'PHASE2', 'PHASE3', 'PHASE2', 'PHAS...","['COMPLETED', 'COMPLETED', 'COMPLETED', 'COMPL...",...,2005.0,Temozolomide,glioblastoma,1999.0,glioblastoma; newly diagnosed glioblastoma,"['ANDA201528', 'ANDA201742', 'ANDA204639', 'AN...",glioblastoma <> temozolomide,True,NaN,glioblastoma


# Filter for neuro pairs

### neuro filter 17

In [17]:
translation_table = pd.concat([translation_table_full, fda_only_translated], ignore_index=True)

In [18]:
disease_keywords = {
    "multiple_sclerosis": ["multiple sclerosis"],
    "alzheimers_disease": ["alzheimer"],
    "parkinsons_disease": ["parkinson"],
    "huntingtons_disease": ["huntington"],
    "motor_neuron_disease": ["motor neuron", "amyotrophic lateral sclerosis", "als"],
    "neuropathic_pain": ["neuropathic pain", "neuralgia", "neuropathy"],
    "prion_disease": ["prion", "creutzfeldt", "cjd"],
    "spinal_cord_injury": ["spinal cord injury"],
    "traumatic_brain_injury": ["traumatic brain injury", "tbi"],
    "subarachnoid_hemorrhage": ["subarachnoid hemorrhage"],
    "stroke": ["stroke", "cerebral infarction", "brain ischem"],
    "schizophrenia": ["schizophrenia", "psychosis"],
    "depression": ["depression", "major depressive"],
    "addiction": ["addiction", "substance use", "dependence"],
    "autism": ["autism", "autism spectrum"],
    "brain_tumor": ["brain tumor", "glioblastoma", "glioma", "meningioma"],
    "epilepsy": ["epilepsy", "seizure"]
}

df = translation_table.copy()
df["disease_part"] = (
    df["normalized_key"]
    .str.lower()
    .str.split("<>")
    .str[0]
    .str.strip()
)

def match_disease(text):
    for label, keywords in disease_keywords.items():
        for kw in keywords:
            if kw in text:
                return label
    return None

df["matched_disease"] = df["disease_part"].apply(match_disease)

translation_table = df[df["matched_disease"].notna()]

n_diseases = translation_table["merged_mondo_label"].nunique()
n_drugs = translation_table["merged_umls_label"].nunique()

print("\n=== Entity coverage summary ===")
print(f"Translation table shape: {translation_table.shape,}")
print(f"Total unique diseases: {n_diseases:,}")
print(f"Total unique drugs: {n_drugs:,}")


=== Entity coverage summary ===
Translation table shape: ((7223, 32),)
Total unique diseases: 242
Total unique drugs: 2,613


In [19]:
translation_table.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_NEURO_17.csv", index=False)

### neuro filter library

In [20]:
translation_table = pd.concat([translation_table_full, fda_only_translated], ignore_index=True)

In [21]:
neuro = pd.read_csv("./data/list_neuro_conditions.csv")
disease_terms = (
    neuro["disease_mondo_term_norm"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

terms_set = set(disease_terms)
terms_list = list(terms_set)
terms = sorted({
    t.strip().lower()
    for t in terms_list
    if isinstance(t, str) and t.strip()
}, key=len, reverse=True)

terms = [t for t in terms if len(t) > 4]
pattern = re.compile("|".join(map(re.escape, terms)))

s = (
    translation_table["normalized_key"]
    .fillna("")
    .astype(str)
    .str.lower()
)

terms_l = [t.lower() for t in terms]  # ensure terms are lowercase too

matched = np.full(len(s), None, dtype=object)
mask = np.zeros(len(s), dtype=bool)

chunk_size = 500

for i in tqdm(range(0, len(terms_l), chunk_size)):
    chunk_terms = terms_l[i : i + chunk_size]

    # Build alternation regex for this chunk
    pattern_str = "|".join(map(re.escape, chunk_terms))

    # Extract first match from this chunk (per row); NaN if none
    m = s.str.extract(f"({pattern_str})", expand=False)

    hit = m.notna().to_numpy()

    # update overall mask
    mask |= hit

    # only fill matched term where we don't already have one
    fill_idx = hit & (matched == None)
    matched[fill_idx] = m.to_numpy()[fill_idx]

translation_table["matched_term"] = matched
translation_table = translation_table[mask]

translation_table.shape


  0%|          | 0/20 [00:00<?, ?it/s]

(17158, 31)

In [22]:
n_diseases = translation_table["merged_mondo_label"].nunique()
n_drugs = translation_table["merged_umls_label"].nunique()

print("\n=== Entity coverage summary ===")
print(f"Translation table shape: {translation_table.shape,}")
print(f"Total unique diseases: {n_diseases:,}")
print(f"Total unique drugs: {n_drugs:,}")


=== Entity coverage summary ===
Translation table shape: ((17158, 31),)
Total unique diseases: 827
Total unique drugs: 3,974


In [23]:
translation_table.normalized_key.nunique()

17158

In [24]:
translation_table.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_NEURO.csv", index=False)